# Join station features to weather
This notebook only defines `add_weather()`. It does not collect data, train models, or write tables.
Load it from a sibling Databricks notebook using a separate cell:

```python
%run ./12_join_station_weather
```

Then, after building bike features and before selecting model columns:

```python
bike_features = add_weather(bike_features)
```

Requires the mapping and forecast-history tables created by notebook 11 in the same workspace.
Input: a Spark dataframe with unique `(station_id, fetched_at)` keys and no weather columns already attached.
All original columns are retained. Missing mappings or eligible forecasts produce null weather.

For each row, select the latest snapshot retrieved by `fetched_at`, using the hour containing
`fetched_at + 15 minutes`. These are hourly approximations, not exact 15-minute forecasts.
Precipitation/snowfall are accumulation values, not amounts for the next 15 minutes.
Current station mappings are not a history of station relocations. Older rows before collection
began will generally have no forecast. Do not fill missing weather with zeros.

In [ ]:
def add_weather(bike_features):
    """Attach hourly weather to unique station_id/fetched_at rows.

    Input timestamps must represent UTC. Input must not already have weather columns.
    Keep missing forecasts as nulls. The mapping table reflects current station locations.
    """
    spark = bike_features.sparkSession
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    bike_features.createOrReplaceTempView("bike_features_for_weather")

    return spark.sql("""
        WITH ranked_matches AS (
            SELECT
                p.*,
                m.forecast_location_id,
                f.valid_at_utc AS weather_valid_at_utc,
                f.retrieved_at_utc AS weather_retrieved_at_utc,
                f.model AS weather_model,
                f.temperature_2m,
                f.apparent_temperature,
                f.relative_humidity_2m,
                f.precipitation,
                f.snowfall,
                f.wind_speed_10m,
                f.wind_gusts_10m,
                f.cloud_cover,
                f.visibility,
                ROW_NUMBER() OVER (
                    PARTITION BY p.station_id, p.fetched_at
                    ORDER BY f.retrieved_at_utc DESC
                ) AS weather_match_rank
            FROM bike_features_for_weather AS p
            LEFT JOIN workspace.default.station_forecast_mapping AS m
                ON p.station_id = m.station_id
            LEFT JOIN workspace.default.weather_forecast_history AS f
                ON m.forecast_location_id = f.forecast_location_id
               AND f.retrieved_at_utc <= p.fetched_at
               AND f.valid_at_utc = date_trunc(
                   'HOUR', p.fetched_at + INTERVAL 15 MINUTES
               )
               AND f.model = 'gfs_seamless'
        )
        SELECT * EXCEPT (weather_match_rank)
        FROM ranked_matches
        WHERE weather_match_rank = 1
    """)

## Example (run in the calling notebook)
```python
example_bike_features = spark.sql("""
    SELECT station_id, MAX(fetched_at) AS fetched_at
    FROM workspace.default.live_station_status
    GROUP BY station_id
""")

example_with_weather = add_weather(example_bike_features)
display(example_with_weather.limit(5))

display(example_with_weather.selectExpr(
    "COUNT(*) AS rows",
    "COUNT(weather_retrieved_at_utc) AS rows_with_forecast",
    "COUNT_IF(weather_retrieved_at_utc IS NULL) AS rows_without_forecast",
    "COUNT_IF(weather_retrieved_at_utc > fetched_at) AS forecasts_retrieved_too_late"
))
```

For the team pipeline, call `add_weather()` after `build_predictors()` and before `.select(*columns)`.
The historical training join belongs before its feature-column selection too. Do not add weather to
model feature lists until matching training data are available; existing fitted models do not start
using weather merely because these columns are present. Team seed remains 622; this join uses no randomness.